In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import os

In [2]:
torch.cuda.is_available()

True

In [3]:
# ---------------------------------------------------------
# 1. Hyperparameters & Device configuration
# ---------------------------------------------------------
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 50
IMAGE_SIZE = 128 # Resizing all images to 64x64
DATA_DIR = './simpsons_dataset' # Ensure this points to the extracted folder

# Select GPU if available, else Apple Silicon (MPS), else CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# ---------------------------------------------------------
# 2. Data Loading & Preprocessing
# ---------------------------------------------------------
# Data augmentation for training, basic transforms for validation
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), # Randomly tilts the image up to 15 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Randomly alters lighting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the entire dataset
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
NUM_CLASSES = len(full_dataset.classes)
print(f"Total classes: {NUM_CLASSES}")

# Split into Train (80%) and Validation (20%)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Extract labels for just the training subset
train_indices = train_dataset.indices
train_labels = [full_dataset.targets[i] for i in train_indices]

# Count how many images belong to each class
class_sample_counts = np.bincount(train_labels)

# Calculate the weights
total_samples = len(train_labels)
num_classes = len(full_dataset.classes)
class_weights = total_samples / (num_classes * (class_sample_counts + 1e-5))

# Convert to tensor and send to GPU/MPS/CPU
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Class Weights calculated successfully!")

Total classes: 43
Class Weights calculated successfully!


In [5]:
# ---------------------------------------------------------
# 3. Custom CNN Architecture
# ---------------------------------------------------------
class SimpsonsCNN_Deep(nn.Module):
    def __init__(self, num_classes):
        super(SimpsonsCNN_Deep, self).__init__()
        
        # Block 1: 128x128 -> 64x64
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2) 
        
        # Block 2: 64x64 -> 32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2) 
        
        # Block 3: 32x32 -> 16x16
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2) 
        
        # Block 4: 16x16 -> 8x8 (NEW BLOCK)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(2, 2)
        
        # Fully Connected Layers
        # Input: 256 channels * 8 width * 8 height
        self.fc1 = nn.Linear(256 * 8 * 8, 512)
        self.bn5 = nn.BatchNorm1d(512) # 1D batch norm for linear layers
        self.relu5 = nn.ReLU()
        self.dropout = nn.Dropout(0.5) 
        self.fc2 = nn.Linear(512, num_classes)
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.pool4(self.relu4(self.bn4(self.conv4(x))))
        
        # Flatten the tensor safely
        x = x.view(x.size(0), -1)
        
        x = self.relu5(self.bn5(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Instantiate the new model
model = SimpsonsCNN_Deep(num_classes=NUM_CLASSES).to(device)

# ---------------------------------------------------------
# 4. Loss Function & Optimizer
# ---------------------------------------------------------
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# NEW: The Learning Rate Scheduler
# mode='min': We want the validation loss to minimize
# factor=0.5: Cut the learning rate in half when triggered
# patience=3: Wait for 3 epochs of zero improvement before stepping in
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [6]:
# ---------------------------------------------------------
# 5. Training and Evaluation Loop
# ---------------------------------------------------------
def train_model():
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        train_accuracy = 100 * correct_train / total_train
        
        # Validation Phase
        model.eval()
        correct_val = 0
        total_val = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        val_accuracy = 100 * correct_val / total_val
        avg_val_loss = val_loss / len(val_loader)

        # Get the current learning rate to print it
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_accuracy:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}% | "
              f"LR: {current_lr}")
        
        scheduler.step(avg_val_loss)

if __name__ == '__main__':
    train_model()
    # Save the model weights
    torch.save(model.state_dict(), 'simpsons_cnn.pth')
    print("Training complete. Model saved to 'simpsons_cnn.pth'.")

Epoch [1/50] Train Loss: 3.3396, Train Acc: 9.90% | Val Loss: 2.7124, Val Acc: 15.50% | LR: 0.001
Epoch [2/50] Train Loss: 2.5378, Train Acc: 17.48% | Val Loss: 2.2096, Val Acc: 21.45% | LR: 0.001
Epoch [3/50] Train Loss: 2.1093, Train Acc: 21.19% | Val Loss: 1.8779, Val Acc: 24.25% | LR: 0.001
Epoch [4/50] Train Loss: 1.7983, Train Acc: 24.59% | Val Loss: 1.7309, Val Acc: 27.85% | LR: 0.001
Epoch [5/50] Train Loss: 1.6103, Train Acc: 26.49% | Val Loss: 1.6520, Val Acc: 28.33% | LR: 0.001
Epoch [6/50] Train Loss: 1.4262, Train Acc: 28.73% | Val Loss: 1.5217, Val Acc: 30.27% | LR: 0.001
Epoch [7/50] Train Loss: 1.2724, Train Acc: 30.61% | Val Loss: 1.3053, Val Acc: 31.92% | LR: 0.001
Epoch [8/50] Train Loss: 1.1698, Train Acc: 32.26% | Val Loss: 1.4760, Val Acc: 32.04% | LR: 0.001
Epoch [9/50] Train Loss: 1.0802, Train Acc: 32.87% | Val Loss: 1.1827, Val Acc: 35.66% | LR: 0.001
Epoch [10/50] Train Loss: 1.0080, Train Acc: 34.52% | Val Loss: 1.2102, Val Acc: 35.38% | LR: 0.001
Epoch [11/